## Load the existing Intent V2 checkpoint

This loads `output/intent_checkpoint/best.pt` for inference only. Intent V2 is not retrained or changed.


## Colab / project paths

On Colab, put the project folder in Google Drive at `MyDrive/edge_ai` (or change `PROJECT_ROOT` below). It must contain `data/` with the train/test images, train labels, and CSV files. Also copy `output/intent_checkpoint/best.pt` to reuse the existing Intent V2 checkpoint. Checkpoints and outputs are saved under this project folder.


In [1]:
from pathlib import Path
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/edge_ai")
else:
    PROJECT_ROOT = Path.cwd()

# Optional override: set EDGE_AI_ROOT, or edit PROJECT_ROOT above.
if os.environ.get("EDGE_AI_ROOT"):
    PROJECT_ROOT = Path(os.environ["EDGE_AI_ROOT"]).expanduser()


def _is_edge_ai_project(root):
    data_root = Path(root) / "data"
    required_files = (
        data_root / "intent_train.csv",
        data_root / "intent_test.csv",
        data_root / "sample_submission.csv",
    )
    return (
        all(path.is_file() for path in required_files)
        and (data_root / "train" / "images").is_dir()
        and (data_root / "train" / "labels").is_dir()
        and (data_root / "test" / "images").is_dir()
    )


_project_candidates = [PROJECT_ROOT, Path.cwd(), Path.cwd() / "edge_ai"]
_project_candidates.extend(Path.cwd().parents)
_project_candidates.extend([
    Path("/content/drive/MyDrive/edge_ai"),
    Path("/content/drive/MyDrive/Colab Notebooks/edge_ai"),
    Path("/content/edge_ai"),
])
ROOT = next((candidate.resolve() for candidate in _project_candidates if _is_edge_ai_project(candidate)), None)
if ROOT is None:
    raise FileNotFoundError(
        "Project data was not found. Put the project folder in MyDrive/edge_ai "
        "with data/intent_train.csv, data/train/images/, data/train/labels/, "
        "and data/test/images/, or edit PROJECT_ROOT to the folder containing data/."
    )
DATA = ROOT / "data"
INTENT_V2_CHECKPOINT_PATH = ROOT / "output" / "intent_checkpoint" / "best.pt"
if not INTENT_V2_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Missing existing Intent V2 checkpoint: {INTENT_V2_CHECKPOINT_PATH}. "
        "Copy the trained v2 best.pt into output/intent_checkpoint/best.pt."
    )
print(f"Project root: {ROOT}")
print(f"Dataset root: {DATA}")
print(f"Intent V2 checkpoint: {INTENT_V2_CHECKPOINT_PATH}")


Project root: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai
Dataset root: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\data
Intent V2 checkpoint: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\output\intent_checkpoint\best.pt


In [2]:
"""Intent v2 source embedded into train_detector_v3_intent_v2.ipynb."""
from collections import Counter
from pathlib import Path
import random
import re
import string

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset

ROOT = ROOT.resolve()
DATA = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
CHECKPOINTS_DIR = ROOT / "checkpoints"
INTENT_CHECKPOINT_DIR = OUTPUT_DIR / "intent_checkpoint"
TEACHER_CHECKPOINT_DIR = CHECKPOINTS_DIR / "teacher_yolo"
STUDENT_CHECKPOINT_DIR = CHECKPOINTS_DIR / "student_detector_v3"
for directory in (OUTPUT_DIR, INTENT_CHECKPOINT_DIR, TEACHER_CHECKPOINT_DIR, STUDENT_CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
INTENT_BEST_CHECKPOINT = INTENT_CHECKPOINT_DIR / "best.pt"
INTENT_LAST_CHECKPOINT = INTENT_CHECKPOINT_DIR / "last.pt"
STUDENT_BEST_CHECKPOINT = STUDENT_CHECKPOINT_DIR / "best.pt"
STUDENT_LAST_CHECKPOINT = STUDENT_CHECKPOINT_DIR / "last.pt"


def _load_checkpoint_file(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)

INTENT_NAMES = ["COUNT", "EXIST", "LIST", "LOCATION", "OUT_OF_SCOPE"]
INTENT_TO_ID = {name: i for i, name in enumerate(INTENT_NAMES)}
WORD_VOCAB_SIZE, WORD_DIM = 512, 96
CHAR_VOCAB_SIZE, CHAR_DIM = 40, 24
WORD_FILTERS, CHAR_FILTERS = 64, 32
FUSION_DIM = 150
MAX_WORDS, MAX_CHARS = 32, 128
INTENT_PARAMS_EXPECTED = 199_852

OBJECT_ALIASES = [
    ("xe oto do choi", "toy_car_obj"), ("o to do choi", "toy_car_obj"),
    ("oto do choi", "toy_car_obj"), ("xe do choi", "toy_car_obj"),
    ("dien thoai di dong", "phone_obj"), ("smartphone", "phone_obj"),
    ("dien thoai", "phone_obj"), ("phone", "phone_obj"),
    ("chuot may tinh", "mouse_obj"), ("chuot", "mouse_obj"),
    ("quyen sach", "book_obj"), ("cuon sach", "book_obj"), ("sach", "book_obj"),
    ("binh nuoc", "bottle_obj"), ("chai nuoc", "bottle_obj"),
    ("chai nhua", "bottle_obj"), ("chai", "bottle_obj"),
    ("coc uong nuoc", "cup_obj"), ("coc", "cup_obj"),
    ("tach", "cup_obj"), ("ly", "cup_obj"),
    ("thung giay", "box_obj"), ("hop giay", "box_obj"), ("hop", "box_obj"),
]
POLITE_PREFIXES = (
    "cho minh hoi", "cho toi hoi", "ban oi", "xin hoi", "xin cho hoi",
    "xin hoi", "hay cho toi hoi", "hay cho minh hoi",
)


def clean_intent_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9_\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def canonical_intent_text(text):
    text = clean_intent_text(text)
    for phrase, canonical in sorted(OBJECT_ALIASES, key=lambda item: len(item[0]), reverse=True):
        text = re.sub(r"\b" + re.escape(phrase) + r"\b", canonical, text)
    return re.sub(r"\s+", " ", text).strip()


def intent_template_key(text):
    text = canonical_intent_text(text)
    for phrase in POLITE_PREFIXES:
        text = re.sub(r"\b" + re.escape(phrase) + r"\b", " ", text)
    # Collapse image-scene wording while retaining operation and entity cues.
    text = re.sub(r"\b(trong|buc|khung|hinh|anh|canh|nay|kia|day)\b", " IMG ", text)
    text = re.sub(r"\b\d+\b", " NUM ", text)
    return re.sub(r"\s+", " ", text).strip()


def make_intent_vocab(texts):
    counts = Counter(token for text in texts for token in canonical_intent_text(text).split())
    words = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(WORD_VOCAB_SIZE - 2)]
    words += [f"<extra_{i}>" for i in range(WORD_VOCAB_SIZE - len(words))]
    return {word: i for i, word in enumerate(words)}


CHAR_ALPHABET = string.ascii_lowercase + string.digits + "_ "
CHAR_TO_ID = {"<pad>": 0, "<unk>": 1}
CHAR_TO_ID.update({char: i + 2 for i, char in enumerate(CHAR_ALPHABET)})
assert len(CHAR_TO_ID) == CHAR_VOCAB_SIZE


def encode_words(text, vocab):
    ids = [vocab.get(token, 1) for token in canonical_intent_text(text).split()[:MAX_WORDS]]
    return ids + [0] * (MAX_WORDS - len(ids))


def encode_chars(text):
    chars = [CHAR_TO_ID.get(char, 1) for char in clean_intent_text(text)[:MAX_CHARS]]
    return chars + [0] * (MAX_CHARS - len(chars))


class IntentDataset(Dataset):
    def __init__(self, frame, vocab):
        self.words = torch.tensor([encode_words(q, vocab) for q in frame.question], dtype=torch.long)
        self.chars = torch.tensor([encode_chars(q) for q in frame.question], dtype=torch.long)
        self.labels = torch.tensor([INTENT_TO_ID[y] for y in frame.intent], dtype=torch.long)
        self.scope = (self.labels != INTENT_TO_ID["OUT_OF_SCOPE"]).long()
        self.operation = self.labels.clone()
        self.operation[self.labels == INTENT_TO_ID["OUT_OF_SCOPE"]] = -100
        self.operation[self.operation >= INTENT_TO_ID["OUT_OF_SCOPE"]] = -100

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.words[index], self.chars[index], self.labels[index], self.scope[index], self.operation[index]


class IntentTextCNNV2(nn.Module):
    """Word/character TextCNN with separate scope and supported-action heads."""
    def __init__(self):
        super().__init__()
        self.word_embedding = nn.Embedding(WORD_VOCAB_SIZE, WORD_DIM, padding_idx=0)
        self.char_embedding = nn.Embedding(CHAR_VOCAB_SIZE, CHAR_DIM, padding_idx=0)
        self.word_convs = nn.ModuleList([nn.Conv1d(WORD_DIM, WORD_FILTERS, k) for k in (2, 3, 4, 5)])
        self.char_convs = nn.ModuleList([nn.Conv1d(CHAR_DIM, CHAR_FILTERS, k) for k in (3, 4, 5)])
        self.fusion = nn.Linear(WORD_FILTERS * 4 + CHAR_FILTERS * 3, FUSION_DIM)
        self.norm = nn.LayerNorm(FUSION_DIM)
        self.dropout = nn.Dropout(0.4)
        self.scope_head = nn.Linear(FUSION_DIM, 2)  # 0=OOS, 1=in-scope
        self.operation_head = nn.Linear(FUSION_DIM, 4)  # COUNT, EXIST, LIST, LOCATION

    @staticmethod
    def _masked_pool(conv, embedded, mask):
        width = conv.kernel_size[0]
        values = F.relu(conv(embedded.transpose(1, 2)))
        valid = mask.unfold(1, width, 1).any(dim=-1)
        values = values.masked_fill(~valid.unsqueeze(1), -1e4)
        return values.max(dim=-1).values

    def forward(self, words, chars):
        word_mask, char_mask = words.ne(0), chars.ne(0)
        w = self.word_embedding(words)
        c = self.char_embedding(chars)
        pooled = [self._masked_pool(conv, w, word_mask) for conv in self.word_convs]
        pooled += [self._masked_pool(conv, c, char_mask) for conv in self.char_convs]
        features = torch.cat(pooled, dim=1)
        features = self.dropout(self.norm(F.relu(self.fusion(features))))
        return self.scope_head(features), self.operation_head(features)


def count_intent_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def make_grouped_intent_split(frame, seed=42):
    groups = frame.question.map(intent_template_key).to_numpy()
    labels = frame.intent.to_numpy()
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    train_idx, val_idx = next(splitter.split(frame.question, labels, groups))
    train_frame, val_frame = frame.iloc[train_idx].reset_index(drop=True), frame.iloc[val_idx].reset_index(drop=True)
    train_groups = set(groups[train_idx])
    val_groups = set(groups[val_idx])
    overlap = len(train_groups & val_groups)
    if overlap:
        raise RuntimeError(f"Template leakage in grouped split: {overlap} groups overlap")
    return train_frame, val_frame


def train_intent_v2(max_epochs=80, patience_limit=10, batch_size=64, seed=42):
    best_path, last_path = INTENT_BEST_CHECKPOINT, INTENT_LAST_CHECKPOINT
    resume_state = _load_checkpoint_file(last_path) if last_path.exists() else None
    if resume_state is not None and resume_state.get("training_stage") != "intent_v2":
        raise RuntimeError(f"{last_path} belongs to another training stage.")
    if resume_state is None and best_path.exists():
        print("Intent V2 already has a trained best checkpoint; leaving it unchanged:", best_path)
        return best_path
    if resume_state is not None and (int(resume_state.get("epoch", 0)) >= max_epochs or int(resume_state.get("patience", 0)) >= patience_limit):
        print("Intent V2 last checkpoint already reached its stopping condition:", last_path)
        return best_path
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    frame = pd.read_csv(DATA / "intent_train.csv")
    train_frame, val_frame = make_grouped_intent_split(frame, seed)
    vocab = resume_state["word_vocab"] if resume_state is not None else make_intent_vocab(train_frame.question)
    train_loader = DataLoader(IntentDataset(train_frame, vocab), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(IntentDataset(val_frame, vocab), batch_size=batch_size, shuffle=False)
    model = IntentTextCNNV2().to(device)
    nparams = count_intent_parameters(model)
    assert nparams == INTENT_PARAMS_EXPECTED, (nparams, INTENT_PARAMS_EXPECTED)
    print(f"device={device} train={len(train_frame)} val={len(val_frame)} groups={len(set(train_frame.question.map(intent_template_key)))} params={nparams}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=2e-5)
    if resume_state is not None:
        model.load_state_dict(resume_state["model"])
        optimizer.load_state_dict(resume_state["optimizer"])
        scheduler.load_state_dict(resume_state["scheduler"])
        best_f1 = float(resume_state.get("best_f1", -1.0))
        best_epoch = int(resume_state.get("best_epoch", 0))
        patience = int(resume_state.get("patience", 0))
        history = list(resume_state.get("history", []))
        start_epoch = int(resume_state["epoch"]) + 1
        print(f"Resuming Intent V2 at epoch {start_epoch - 1}:", last_path)
    else:
        best_f1, best_epoch, patience, history, start_epoch = -1.0, 0, 0, [], 1
    for epoch in range(start_epoch, max_epochs + 1):
        model.train(); total_loss = 0.0
        for words, chars, labels, scope, operation in train_loader:
            words, chars = words.to(device), chars.to(device)
            labels, scope, operation = labels.to(device), scope.to(device), operation.to(device)
            scope_logits, op_logits = model(words, chars)
            loss_scope = F.cross_entropy(scope_logits, scope)
            valid = operation.ne(-100)
            loss_op = F.cross_entropy(op_logits[valid], operation[valid]) if valid.any() else op_logits.sum() * 0.0
            loss = loss_scope + loss_op
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0); optimizer.step()
            total_loss += float(loss.detach()) * len(words)
        model.eval(); predictions, gold = [], []
        with torch.no_grad():
            for words, chars, labels, _, _ in val_loader:
                scope_logits, op_logits = model(words.to(device), chars.to(device))
                scope_pred, op_pred = scope_logits.argmax(1).cpu(), op_logits.argmax(1).cpu()
                predictions.extend(torch.where(scope_pred == 0, INTENT_TO_ID["OUT_OF_SCOPE"], op_pred).tolist())
                gold.extend(labels.tolist())
        acc = accuracy_score(gold, predictions)
        macro_f1 = f1_score(gold, predictions, average="macro")
        scheduler.step()
        row = {"epoch": epoch, "train_loss": total_loss / len(train_frame), "val_accuracy": acc, "val_macro_f1": macro_f1}
        history.append(row)
        if macro_f1 > best_f1 + 1e-5:
            best_f1, best_epoch, patience = macro_f1, epoch, 0
            torch.save({"model": model.state_dict(), "word_vocab": vocab, "char_vocab": CHAR_TO_ID,
                        "intent_names": INTENT_NAMES, "params": nparams, "epoch": epoch,
                        "val_accuracy": acc, "val_macro_f1": macro_f1, "template_split": True,
                        "training_stage": "intent_v2"}, best_path)
        else:
            patience += 1
        last_state = {"model": model.state_dict(), "word_vocab": vocab, "char_vocab": CHAR_TO_ID,
                      "intent_names": INTENT_NAMES, "params": nparams, "epoch": epoch,
                      "val_accuracy": acc, "val_macro_f1": macro_f1, "template_split": True,
                      "training_stage": "intent_v2", "best_f1": best_f1, "best_epoch": best_epoch,
                      "patience": patience, "history": history,
                      "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict()}
        torch.save(last_state, last_path)
        pd.DataFrame(history).to_csv(OUTPUT_DIR / "intent_v2_history.csv", index=False)
        print(f"{epoch:03d}: loss={row['train_loss']:.4f} val_acc={acc:.4f} macro_f1={macro_f1:.4f} best={best_f1:.4f}@{best_epoch} patience={patience}/{patience_limit}")
        if patience >= patience_limit: break
    best_state = _load_checkpoint_file(best_path)
    model.load_state_dict(best_state["model"]); model.eval()
    best_predictions, best_gold = [], []
    with torch.no_grad():
        for words, chars, labels, _, _ in val_loader:
            scope_logits, op_logits = model(words.to(device), chars.to(device))
            scope_pred, op_pred = scope_logits.argmax(1).cpu(), op_logits.argmax(1).cpu()
            best_predictions.extend(torch.where(scope_pred == 0, INTENT_TO_ID["OUT_OF_SCOPE"], op_pred).tolist())
            best_gold.extend(labels.tolist())
    print(classification_report(best_gold, best_predictions, labels=list(range(5)), target_names=INTENT_NAMES, zero_division=0))
    print(f"saved {best_path} epoch={best_epoch} macro_f1={best_f1:.4f}; resumable last={last_path}")
    return best_path


In [3]:
"""Detector v3 source embedded into train_detector_v3_intent_v2.ipynb."""
import csv
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from torch.utils.data import DataLoader, Dataset
from torchvision.ops import batched_nms

ROOT = ROOT.resolve()
DATA = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
CHECKPOINTS_DIR = ROOT / "checkpoints"
INTENT_CHECKPOINT_DIR = OUTPUT_DIR / "intent_checkpoint"
TEACHER_CHECKPOINT_DIR = CHECKPOINTS_DIR / "teacher_yolo"
STUDENT_CHECKPOINT_DIR = CHECKPOINTS_DIR / "student_detector_v3"
for directory in (OUTPUT_DIR, INTENT_CHECKPOINT_DIR, TEACHER_CHECKPOINT_DIR, STUDENT_CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
INTENT_BEST_CHECKPOINT = INTENT_CHECKPOINT_DIR / "best.pt"
INTENT_LAST_CHECKPOINT = INTENT_CHECKPOINT_DIR / "last.pt"
STUDENT_BEST_CHECKPOINT = STUDENT_CHECKPOINT_DIR / "best.pt"
STUDENT_LAST_CHECKPOINT = STUDENT_CHECKPOINT_DIR / "last.pt"

NC = 7
CLASS_NAMES = ["bottle", "cup", "phone", "book", "mouse", "box", "toy_car"]
IMG_W, IMG_H = 320, 240
STRIDES = (4, 8, 16)
LEVEL_MAX_SIZE = (64.0, 128.0, float("inf"))
CENTER_RADIUS = 1.5
REG_MAX = 24
DET_PARAMS_EXPECTED = 199_416


class DetDatasetV3(Dataset):
    def __init__(self, paths, augment=False):
        self.paths = list(paths)
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = Image.open(path).convert("RGB").resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)
        rows = []
        label_path = DATA / "train" / "labels" / f"{path.stem}.txt"
        for line in label_path.read_text(encoding="utf-8").splitlines():
            values = line.split()
            if len(values) == 5:
                cls, cx, cy, width, height = int(values[0]), *map(float, values[1:])
                if 0 <= cls < NC and width > 0 and height > 0:
                    rows.append([cls, cx, cy, width, height])

        if self.augment and random.random() < 0.55:
            scale = random.uniform(1.0, 1.65)
            if scale > 1.02:
                scaled_w, scaled_h = round(IMG_W * scale), round(IMG_H * scale)
                image = image.resize((scaled_w, scaled_h), Image.Resampling.BILINEAR)
                left = random.randint(0, scaled_w - IMG_W)
                top = random.randint(0, scaled_h - IMG_H)
                image = image.crop((left, top, left + IMG_W, top + IMG_H))
                zoomed = []
                for cls, cx, cy, width, height in rows:
                    x1, y1 = cx * scaled_w - left - width * scaled_w / 2, cy * scaled_h - top - height * scaled_h / 2
                    x2, y2 = cx * scaled_w - left + width * scaled_w / 2, cy * scaled_h - top + height * scaled_h / 2
                    old_area = max((x2 - x1) * (y2 - y1), 1e-6)
                    ix1, iy1 = max(0.0, x1), max(0.0, y1)
                    ix2, iy2 = min(float(IMG_W), x2), min(float(IMG_H), y2)
                    visible = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1) / old_area
                    if visible >= 0.4 and ix2 > ix1 and iy2 > iy1:
                        zoomed.append([cls, (ix1 + ix2) / (2 * IMG_W), (iy1 + iy2) / (2 * IMG_H),
                                       (ix2 - ix1) / IMG_W, (iy2 - iy1) / IMG_H])
                rows = zoomed

        if self.augment and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            rows = [[cls, 1.0 - cx, cy, width, height] for cls, cx, cy, width, height in rows]
        if self.augment and random.random() < 0.25:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.85, 1.15))
        if self.augment and random.random() < 0.25:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.85, 1.15))

        array = np.asarray(image, dtype=np.uint8).copy()
        tensor = torch.from_numpy(array.transpose(2, 0, 1)).float().div_(255.0)
        labels = torch.tensor(rows, dtype=torch.float32).reshape(-1, 5)
        return tensor, labels


def det_collate(batch):
    images, labels = zip(*batch)
    return torch.stack(images), list(labels)


class DetConvBNAct(nn.Module):
    def __init__(self, ci, co, kernel=3, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(ci, co, kernel, stride, kernel // 2, bias=False)
        self.bn = nn.BatchNorm2d(co)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class DetDSBlock(nn.Module):
    def __init__(self, ci, co, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(ci, ci, 3, stride, 1, groups=ci, bias=False)
        self.dw_bn = nn.BatchNorm2d(ci)
        self.pw = nn.Conv2d(ci, co, 1, bias=False)
        self.pw_bn = nn.BatchNorm2d(co)
        self.skip = (nn.Sequential(nn.Conv2d(ci, co, 1, stride, bias=False), nn.BatchNorm2d(co))
                     if ci != co or stride != 1 else nn.Identity())

    def forward(self, x):
        y = F.silu(self.dw_bn(self.dw(x)))
        y = self.pw_bn(self.pw(y))
        return F.silu(y + self.skip(x))


def det_stack(channels, repeats):
    return nn.Sequential(*[DetDSBlock(channels, channels) for _ in range(repeats)])


class TinyMultiScaleDetV3(nn.Module):
    """Three-level P2/P3/P4 detector with reg_max=24 distributional LTRB."""
    def __init__(self):
        super().__init__()
        self.stem = DetConvBNAct(3, 24, 3, 2)
        self.s1 = nn.Sequential(DetDSBlock(24, 32, 2), DetDSBlock(32, 32))
        self.s2 = nn.Sequential(DetDSBlock(32, 48, 2), DetDSBlock(48, 48), DetDSBlock(48, 48))
        self.s3 = nn.Sequential(DetDSBlock(48, 64, 2), DetDSBlock(64, 64), DetDSBlock(64, 80))
        self.s4 = nn.Sequential(DetDSBlock(80, 96, 2), DetDSBlock(96, 96))
        width = 80
        self.p5_lateral = DetConvBNAct(96, width, 1)
        self.p4_lateral = DetConvBNAct(80, width, 1)
        self.p3_lateral = DetConvBNAct(48, width, 1)
        self.p2_lateral = DetConvBNAct(32, width, 1)
        self.f4, self.f3, self.f2 = det_stack(width, 2), det_stack(width, 2), det_stack(width, 3)
        # Three 80->80 blocks are reallocated from the prediction towers to DFL bins.
        self.h4, self.h3, self.h2 = det_stack(width, 1), det_stack(width, 2), det_stack(width, 2)
        out_channels = NC + 1 + 4 * REG_MAX
        self.out4 = nn.Conv2d(width, out_channels, 1)
        self.out3 = nn.Conv2d(width, out_channels, 1)
        self.out2 = nn.Conv2d(width, out_channels, 1)
        self._init_heads()

    def _init_heads(self):
        bins = torch.arange(REG_MAX, dtype=torch.float32)
        for head in (self.out2, self.out3, self.out4):
            nn.init.normal_(head.weight, mean=0.0, std=0.01)
            nn.init.zeros_(head.bias)
            with torch.no_grad():
                head.bias[:NC].fill_(-4.0)
                head.bias[NC:NC + 4 * REG_MAX].copy_(-torch.abs(bins - 3.0).repeat(4))
                head.bias[NC + 4 * REG_MAX] = -2.0

    def forward(self, x):
        x = self.stem(x)
        c2 = self.s1(x)       # stride 4
        c3 = self.s2(c2)      # stride 8
        c4 = self.s3(c3)      # stride 16
        c5 = self.s4(c4)      # stride 32
        p4 = self.f4(self.p4_lateral(c4) + F.interpolate(self.p5_lateral(c5), size=c4.shape[-2:], mode="nearest"))
        p3 = self.f3(self.p3_lateral(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="nearest"))
        p2 = self.f2(self.p2_lateral(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="nearest"))
        return self.out2(self.h2(p2)), self.out3(self.h3(p3)), self.out4(self.h4(p4))


def count_det_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def expected_ltrb(raw, stride):
    batch, _, height, width = raw.shape
    dist_logits = raw[:, NC:NC + 4 * REG_MAX].reshape(batch, 4, REG_MAX, height, width)
    bins = torch.arange(REG_MAX, device=raw.device, dtype=torch.float32).view(1, 1, REG_MAX, 1, 1)
    return (dist_logits.float().softmax(dim=2) * bins).sum(dim=2) * stride


def build_det_targets(labels, outputs, device):
    targets, area_maps, point_maps = [], [], []
    for output, stride in zip(outputs, STRIDES):
        batch, _, height, width = output.shape
        targets.append({"cls": torch.zeros((batch, NC, height, width), device=device),
                        "reg": torch.zeros((batch, 4, height, width), device=device),
                        "ctr": torch.zeros((batch, height, width), device=device),
                        "pos": torch.zeros((batch, height, width), dtype=torch.bool, device=device)})
        area_maps.append(torch.full((batch, height, width), float("inf"), device=device))
        px = (torch.arange(width, device=device, dtype=torch.float32) + 0.5) * stride
        py = (torch.arange(height, device=device, dtype=torch.float32) + 0.5) * stride
        point_maps.append((px[None, :].expand(height, width), py[:, None].expand(height, width)))

    for batch_index, rows in enumerate(labels):
        rows = rows.to(device=device, dtype=torch.float32)
        for row in rows:
            cls = int(row[0].item())
            cx, cy, bw, bh = row[1] * IMG_W, row[2] * IMG_H, row[3] * IMG_W, row[4] * IMG_H
            x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
            level = 0 if torch.maximum(bw, bh) <= LEVEL_MAX_SIZE[0] else 1 if torch.maximum(bw, bh) <= LEVEL_MAX_SIZE[1] else 2
            stride = STRIDES[level]
            px, py = point_maps[level]
            inside_box = (px >= x1) & (px <= x2) & (py >= y1) & (py <= y2)
            radius = CENTER_RADIUS * stride
            cx1, cy1 = torch.maximum(x1, cx - radius), torch.maximum(y1, cy - radius)
            cx2, cy2 = torch.minimum(x2, cx + radius), torch.minimum(y2, cy + radius)
            center_region = (px >= cx1) & (px <= cx2) & (py >= cy1) & (py <= cy2)
            update = inside_box & center_region & (bw * bh < area_maps[level][batch_index])
            if not update.any():
                continue
            left, top, right, bottom = px - x1, py - y1, x2 - px, y2 - py
            target = targets[level]
            target["cls"][batch_index, :, update] = 0.0
            target["cls"][batch_index, cls, update] = 1.0
            target["reg"][batch_index, :, update] = torch.stack((left, top, right, bottom), dim=0)[:, update] / stride
            lr = torch.minimum(left, right) / torch.maximum(left, right).clamp_min(1e-6)
            tb = torch.minimum(top, bottom) / torch.maximum(top, bottom).clamp_min(1e-6)
            target["ctr"][batch_index, update] = torch.sqrt((lr * tb).clamp_min(0.0))[update]
            target["pos"][batch_index, update] = True
            area_maps[level][batch_index, update] = bw * bh
    return targets


def detector_v3_loss(outputs, targets):
    num_pos = max(sum(int(t["pos"].sum().item()) for t in targets), 1)
    cls_loss = outputs[0].new_zeros((), dtype=torch.float32)
    ctr_loss = outputs[0].new_zeros((), dtype=torch.float32)
    giou_loss = outputs[0].new_zeros((), dtype=torch.float32)
    dfl_loss = outputs[0].new_zeros((), dtype=torch.float32)
    for raw, target, stride in zip(outputs, targets, STRIDES):
        logits = raw[:, :NC].float()
        ctr_logits = raw[:, NC + 4 * REG_MAX].float()
        cls_target, positive = target["cls"], target["pos"]
        prob = logits.sigmoid()
        bce = F.binary_cross_entropy_with_logits(logits, cls_target, reduction="none")
        p_t = prob * cls_target + (1.0 - prob) * (1.0 - cls_target)
        alpha = 0.25 * cls_target + 0.75 * (1.0 - cls_target)
        cls_loss = cls_loss + (alpha * (1.0 - p_t).pow(2) * bce).sum() / num_pos
        if not positive.any():
            continue

        ctr_target = target["ctr"][positive]
        point_weight = ctr_target.clamp_min(0.05)
        ctr_loss = ctr_loss + F.binary_cross_entropy_with_logits(ctr_logits[positive], ctr_target, reduction="sum") / num_pos
        dist_logits = raw[:, NC:NC + 4 * REG_MAX].float().reshape(raw.shape[0], 4, REG_MAX, raw.shape[-2], raw.shape[-1])
        dist_logits = dist_logits.permute(0, 3, 4, 1, 2)[positive]
        truth = target["reg"].permute(0, 2, 3, 1)[positive].clamp(0.0, REG_MAX - 1.001)
        lo = truth.floor().long()
        hi = (lo + 1).clamp(max=REG_MAX - 1)
        w_hi = truth - lo.float()
        w_lo = 1.0 - w_hi
        log_prob = dist_logits.log_softmax(dim=-1)
        per_side = -(log_prob.gather(-1, lo.unsqueeze(-1)).squeeze(-1) * w_lo
                     + log_prob.gather(-1, hi.unsqueeze(-1)).squeeze(-1) * w_hi)
        dfl_each = per_side.mean(dim=-1)
        dfl_loss = dfl_loss + (dfl_each * point_weight).sum() / point_weight.sum().clamp_min(1e-6)

        pred = expected_ltrb(raw, stride).permute(0, 2, 3, 1)[positive]
        pl, pt, pr, pb = pred.unbind(-1)
        tl, tt, tr, tb = (truth * stride).unbind(-1)
        inter = (torch.minimum(pl, tl) + torch.minimum(pr, tr)).clamp_min(0) * (torch.minimum(pt, tt) + torch.minimum(pb, tb)).clamp_min(0)
        pred_area, true_area = (pl + pr) * (pt + pb), (tl + tr) * (tt + tb)
        union = pred_area + true_area - inter
        iou = inter / union.clamp_min(1e-6)
        enclosure = ((torch.maximum(pl, tl) + torch.maximum(pr, tr)) * (torch.maximum(pt, tt) + torch.maximum(pb, tb))).clamp_min(1e-6)
        giou = iou - (enclosure - union) / enclosure
        giou_loss = giou_loss + ((1.0 - giou) * point_weight).sum() / point_weight.sum().clamp_min(1e-6)
    return cls_loss + 0.5 * ctr_loss + 2.0 * giou_loss + 0.25 * dfl_loss


@torch.no_grad()
def decode_det_v3(outputs, conf=0.001, pre_nms=1000, max_det=100):
    result = []
    for batch_index in range(outputs[0].shape[0]):
        boxes_all, scores_all, classes_all = [], [], []
        for raw_batch, stride in zip(outputs, STRIDES):
            raw = raw_batch[batch_index].float()
            _, height, width = raw.shape
            cls_scores, cls_ids = raw[:NC].sigmoid().max(dim=0)
            scores = cls_scores * raw[NC + 4 * REG_MAX].sigmoid()
            valid = scores.flatten() >= conf
            if not valid.any():
                continue
            yy, xx = torch.meshgrid(torch.arange(height, device=raw.device, dtype=torch.float32),
                                    torch.arange(width, device=raw.device, dtype=torch.float32), indexing="ij")
            px, py = (xx + 0.5) * stride, (yy + 0.5) * stride
            left, top, right, bottom = expected_ltrb(raw_batch[batch_index:batch_index + 1], stride)[0]
            boxes = torch.stack((px - left, py - top, px + right, py + bottom), dim=-1).reshape(-1, 4)
            ids = torch.nonzero(valid, as_tuple=False).squeeze(1)
            boxes_all.append(boxes[ids]); scores_all.append(scores.flatten()[ids]); classes_all.append(cls_ids.flatten()[ids])
        if not scores_all:
            result.append((torch.empty((0, 4)), torch.empty((0,)), torch.empty((0,), dtype=torch.long)))
            continue
        boxes, scores, classes = torch.cat(boxes_all), torch.cat(scores_all), torch.cat(classes_all)
        if scores.numel() > pre_nms:
            top = scores.topk(pre_nms).indices
            boxes, scores, classes = boxes[top], scores[top], classes[top]
        boxes[:, 0::2].clamp_(0, IMG_W); boxes[:, 1::2].clamp_(0, IMG_H)
        valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
        boxes, scores, classes = boxes[valid], scores[valid], classes[valid]
        if scores.numel():
            keep = batched_nms(boxes, scores, classes, 0.6)[:max_det]
            result.append((boxes[keep].cpu(), scores[keep].cpu(), classes[keep].cpu()))
        else:
            result.append((torch.empty((0, 4)), torch.empty((0,)), torch.empty((0,), dtype=torch.long)))
    return result


def box_iou_one(box, boxes):
    if not boxes:
        return torch.zeros((0,), dtype=torch.float32)
    boxes = torch.stack(boxes)
    lt, rb = torch.maximum(box[:2], boxes[:, :2]), torch.minimum(box[2:], boxes[:, 2:])
    intersection = (rb - lt).clamp_min(0).prod(dim=1)
    area_a = (box[2:] - box[:2]).clamp_min(0).prod()
    area_b = (boxes[:, 2:] - boxes[:, :2]).clamp_min(0).prod(dim=1)
    return intersection / (area_a + area_b - intersection).clamp_min(1e-7)


@torch.no_grad()
def evaluate_det_v3_map50(model, loader, device, return_per_class=False):
    model.eval()
    predictions = [[] for _ in range(NC)]
    ground_truths = [[] for _ in range(NC)]
    image_id = 0
    for images, labels in loader:
        outputs = model(images.to(device, non_blocking=True))
        decoded = decode_det_v3(outputs)
        for batch_index, rows in enumerate(labels):
            gt_for_image = [[] for _ in range(NC)]
            for row in rows.tolist():
                cls, cx, cy, width, height = row
                gt_for_image[int(cls)].append(torch.tensor([
                    (cx-width/2)*IMG_W, (cy-height/2)*IMG_H,
                    (cx+width/2)*IMG_W, (cy+height/2)*IMG_H,
                ]))
            for cls in range(NC):
                ground_truths[cls].append(gt_for_image[cls])
            boxes, scores, classes = decoded[batch_index]
            for box, score, cls in zip(boxes, scores, classes):
                predictions[int(cls)].append((float(score), image_id, box))
            image_id += 1

    class_aps = {}
    for cls in range(NC):
        class_name = CLASS_NAMES[cls]
        total_gt = sum(len(boxes) for boxes in ground_truths[cls])
        if not total_gt:
            class_aps[class_name] = 0.0
            continue
        ranked = sorted(predictions[cls], key=lambda item: item[0], reverse=True)
        matched = [set() for _ in ground_truths[cls]]
        tp, fp = np.zeros(len(ranked), dtype=np.float32), np.zeros(len(ranked), dtype=np.float32)
        for index, (_, im_id, box) in enumerate(ranked):
            gt = ground_truths[cls][im_id]
            if not gt:
                fp[index] = 1.0
                continue
            overlaps = box_iou_one(box, gt)
            best = int(overlaps.argmax())
            if float(overlaps[best]) >= 0.5 and best not in matched[im_id]:
                tp[index] = 1.0
                matched[im_id].add(best)
            else:
                fp[index] = 1.0
        if not ranked:
            class_aps[class_name] = 0.0
            continue
        cumulative_tp, cumulative_fp = np.cumsum(tp), np.cumsum(fp)
        recall = cumulative_tp / total_gt
        precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1e-12)
        class_aps[class_name] = float(np.mean([
            precision[recall >= r].max() if np.any(recall >= r) else 0.0
            for r in np.linspace(0.0, 1.0, 101)
        ]))
    mean_ap = float(np.mean(list(class_aps.values()))) if class_aps else 0.0
    return (mean_ap, class_aps) if return_per_class else mean_ap


## Load the existing Intent V2 checkpoint

This loads `output/intent_checkpoint/best.pt` for inference only. Intent V2 is not retrained or changed.


In [4]:
INTENT_V2_CHECKPOINT = INTENT_BEST_CHECKPOINT
if not INTENT_V2_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing existing Intent V2 checkpoint: {INTENT_V2_CHECKPOINT}. "
        "Copy the trained v2 best.pt to output/intent_checkpoint/best.pt in the project folder."
    )

try:
    intent_v2_state = torch.load(INTENT_V2_CHECKPOINT, map_location="cpu", weights_only=True)
except TypeError:
    intent_v2_state = torch.load(INTENT_V2_CHECKPOINT, map_location="cpu")

intent_v2_model = IntentTextCNNV2()
intent_v2_model.load_state_dict(intent_v2_state["model"])
intent_v2_model.eval()
intent_v2_word_vocab = intent_v2_state["word_vocab"]
intent_v2_char_vocab = intent_v2_state["char_vocab"]
print(
    "Loaded old Intent V2:", INTENT_V2_CHECKPOINT,
    "epoch=", intent_v2_state.get("epoch"),
    "val_accuracy=", intent_v2_state.get("val_accuracy"),
    "params=", count_intent_parameters(intent_v2_model),
)

def predict_intent_v2_existing(text):
    words = torch.tensor([encode_words(text, intent_v2_word_vocab)], dtype=torch.long)
    chars = torch.tensor([encode_chars(text)], dtype=torch.long)
    with torch.inference_mode():
        scope_logits, action_logits = intent_v2_model(words, chars)
        if int(scope_logits.argmax(1).item()) == 0:
            return "OUT_OF_SCOPE"
        return INTENT_NAMES[int(action_logits.argmax(1).item())]

Loaded old Intent V2: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\output\intent_checkpoint\best.pt epoch= 3 val_accuracy= 1.0 params= 199852


## Fine-tune YOLO26m with LoRA

LoRA adapters are added to standard backbone and neck convolutions after Ultralytics builds the seven-class model and transfers pretrained weights. The detection head remains fully trainable for the target classes. Ultralytics validation selects the teacher by mAP@0.5, matching the leaderboard metric; the adapters are merged into the saved teacher before distillation.

In [5]:
import json
import yaml
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer


TEACHER_INIT_WEIGHTS = str(ROOT / "yolo26m.pt") if (ROOT / "yolo26m.pt").exists() else "yolo26m.pt"
TEACHER_RUN_NAME = "teacher_yolo"
TEACHER_RUN_DIR = TEACHER_CHECKPOINT_DIR
TEACHER_WEIGHTS_DIR = TEACHER_RUN_DIR / "weights"
TEACHER_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_BEST_CHECKPOINT = TEACHER_WEIGHTS_DIR / "best.pt"
TEACHER_LAST_CHECKPOINT = TEACHER_WEIGHTS_DIR / "last.pt"
TEACHER_META_PATH = OUTPUT_DIR / "teacher_yolo_meta.json"
TEACHER_STATUS_PATH = OUTPUT_DIR / "teacher_yolo_training_status.json"
TEACHER_MAX_EPOCHS = 100
RESUME_TEACHER_FROM_LAST = False  # Reuse an existing best teacher by default.
TEACHER_IMGSZ = 640
TEACHER_BATCH = 8
TEACHER_SEED = 42
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05


class LoRAConv2d(nn.Module):
    def __init__(self, base_conv, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT):
        super().__init__()
        if base_conv.groups != 1:
            raise ValueError("LoRAConv2d only wraps ungrouped convolutions.")
        self.base = base_conv
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scale = self.alpha / self.rank
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)
        self.lora_A = nn.Conv2d(
            self.base.in_channels, self.rank,
            kernel_size=self.base.kernel_size, stride=self.base.stride,
            padding=self.base.padding, dilation=self.base.dilation,
            bias=False, padding_mode=self.base.padding_mode,
        )
        self.lora_B = nn.Conv2d(self.rank, self.base.out_channels, kernel_size=1, bias=False)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_B.weight)

    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
    @property
    def stride(self): return self.base.stride
    @property
    def padding(self): return self.base.padding
    @property
    def dilation(self): return self.base.dilation
    @property
    def groups(self): return self.base.groups
    @property
    def kernel_size(self): return self.base.kernel_size
    @property
    def in_channels(self): return self.base.in_channels
    @property
    def out_channels(self): return self.base.out_channels

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(self.dropout(x)))

    @torch.no_grad()
    def merged_base(self):
        delta = torch.einsum("or,rihw->oihw", self.lora_B.weight[:, :, 0, 0], self.lora_A.weight)
        self.base.weight.add_(self.scale * delta.to(self.base.weight.dtype))
        return self.base


def inject_yolo_lora(model):
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    if not hasattr(model, "model") or len(model.model) == 0:
        raise RuntimeError("Could not locate YOLO detection head.")
    detection_head = model.model[-1]
    for parameter in detection_head.parameters():
        parameter.requires_grad_(True)
    head_ids = {id(module) for module in detection_head.modules()}
    targets = []

    def replace_convs(parent, prefix=""):
        for child_name, child in list(parent.named_children()):
            full_name = f"{prefix}.{child_name}" if prefix else child_name
            if id(child) in head_ids:
                continue
            if isinstance(child, nn.Conv2d) and child.groups == 1 and child.kernel_size in ((1, 1), (3, 3)):
                setattr(parent, child_name, LoRAConv2d(child))
                targets.append(full_name)
            else:
                replace_convs(child, full_name)

    replace_convs(model)
    if not targets:
        raise RuntimeError("LoRA injection found no eligible backbone/neck convolutions.")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"LoRA layers={len(targets)}; trainable={trainable:,}/{total:,} ({100*trainable/total:.2f}%).")
    return model


def merge_yolo_lora(model):
    merged = 0
    def visit(parent):
        nonlocal merged
        for child_name, child in list(parent.named_children()):
            if isinstance(child, LoRAConv2d):
                setattr(parent, child_name, child.merged_base())
                merged += 1
            else:
                visit(child)
    visit(model)
    return merged


class LoRADetectionTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)
        if any(isinstance(module, LoRAConv2d) for module in model.modules()):
            return model
        return inject_yolo_lora(model)

    def validate(self):
        previous_best = self.best_fitness
        metrics, fitness = super().validate()
        if metrics is None:
            return metrics, fitness
        fitness = float(metrics["metrics/mAP50(B)"])
        self.best_fitness = fitness if previous_best is None else max(previous_best, fitness)
        return metrics, fitness


def detector_train_val_paths(seed=42):
    paths = sorted((DATA / "train" / "images").glob("*.png"))
    random.Random(seed).shuffle(paths)
    n_val = int(round(0.2 * len(paths)))
    return paths[n_val:], paths[:n_val]


def make_teacher_data_yaml(train_paths, val_paths):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    train_list = OUTPUT_DIR / "yolo26m_lora_train_images.txt"
    val_list = OUTPUT_DIR / "yolo26m_lora_val_images.txt"
    train_list.write_text("".join(f"{p.resolve()}\n" for p in train_paths), encoding="utf-8")
    val_list.write_text("".join(f"{p.resolve()}\n" for p in val_paths), encoding="utf-8")
    data_yaml = OUTPUT_DIR / "yolo26m_lora_teacher_data.yaml"
    data_yaml.write_text(yaml.safe_dump({
        "path": str(DATA.resolve()),
        "train": str(train_list.resolve()),
        "val": str(val_list.resolve()),
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }, sort_keys=False), encoding="utf-8")
    return data_yaml


def fine_tune_yolo26m_lora_teacher():
    train_paths, val_paths = detector_train_val_paths(TEACHER_SEED)
    data_yaml = make_teacher_data_yaml(train_paths, val_paths)
    resume = TEACHER_LAST_CHECKPOINT.exists()
    if resume:
        teacher = YOLO(str(TEACHER_LAST_CHECKPOINT))
        print("Resuming YOLO26m LoRA teacher from:", TEACHER_LAST_CHECKPOINT)
    else:
        teacher = YOLO(TEACHER_INIT_WEIGHTS)
        print("Starting YOLO26m LoRA teacher from pretrained weights:", TEACHER_INIT_WEIGHTS)
    TEACHER_STATUS_PATH.unlink(missing_ok=True)
    teacher.train(
        data=str(data_yaml), trainer=LoRADetectionTrainer,
        imgsz=TEACHER_IMGSZ, epochs=TEACHER_MAX_EPOCHS, batch=TEACHER_BATCH,
        device=0 if torch.cuda.is_available() else "cpu", workers=0,
        pretrained=not resume, resume=resume, optimizer="MuSGD",
        lr0=3.8e-4, lrf=0.10, momentum=0.948, weight_decay=2.7e-4,
        warmup_epochs=3.0, nbs=64, cos_lr=True, patience=20,
        save_period=-1, scale=0.50, mosaic=0.75, close_mosaic=15,
        mixup=0.0, copy_paste=0.0, degrees=0.0, translate=0.08,
        shear=0.0, perspective=0.0, fliplr=0.5, flipud=0.0,
        hsv_h=0.015, hsv_s=0.55, hsv_v=0.35, multi_scale=0.0,
        project=str(CHECKPOINTS_DIR), name=TEACHER_RUN_NAME, exist_ok=True,
        seed=TEACHER_SEED, deterministic=True, amp=True, plots=True, val=True,
    )
    if not TEACHER_BEST_CHECKPOINT.exists():
        raise FileNotFoundError(f"Teacher training finished without best checkpoint: {TEACHER_BEST_CHECKPOINT}")
    # Best is merged for inference; last stays unmerged so training can resume its LoRA state.
    best_teacher = YOLO(str(TEACHER_BEST_CHECKPOINT))
    merged_layers = merge_yolo_lora(best_teacher.model)
    if merged_layers:
        best_teacher.save(str(TEACHER_BEST_CHECKPOINT))
    print(f"Teacher best checkpoint ready: {TEACHER_BEST_CHECKPOINT}; merged LoRA layers={merged_layers}")
    TEACHER_META_PATH.write_text(json.dumps({
        "teacher_checkpoint": str(TEACHER_BEST_CHECKPOINT.resolve()),
        "teacher_last_checkpoint": str(TEACHER_LAST_CHECKPOINT.resolve()),
        "source_weights": TEACHER_INIT_WEIGHTS,
        "method": "LoRA on backbone/neck; full detection head fine-tuning",
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "merged_lora_layers": merged_layers, "best_metric": "validation mAP50",
        "epochs_max": TEACHER_MAX_EPOCHS, "early_stopping_patience": 20,
        "imgsz": TEACHER_IMGSZ, "batch": TEACHER_BATCH,
        "optimizer": "MuSGD", "lr0": 3.8e-4, "cos_lr": True,
        "resumed_from_last": resume,
    }, indent=2), encoding="utf-8")
    TEACHER_STATUS_PATH.write_text(json.dumps({
        "completed": True,
        "best_checkpoint": str(TEACHER_BEST_CHECKPOINT.resolve()),
        "last_checkpoint": str(TEACHER_LAST_CHECKPOINT.resolve()),
        "max_epochs": TEACHER_MAX_EPOCHS,
    }, indent=2), encoding="utf-8")
    return TEACHER_BEST_CHECKPOINT



In [6]:
def select_best_existing_lora_teacher():
    status = {}
    if TEACHER_STATUS_PATH.exists():
        try:
            status = json.loads(TEACHER_STATUS_PATH.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            status = {}
    if TEACHER_BEST_CHECKPOINT.exists() and not RESUME_TEACHER_FROM_LAST:
        print("Using existing best YOLO26m LoRA teacher:", TEACHER_BEST_CHECKPOINT)
        return TEACHER_BEST_CHECKPOINT
    if TEACHER_LAST_CHECKPOINT.exists():
        print("Teacher last checkpoint exists; resuming LoRA training from it:", TEACHER_LAST_CHECKPOINT)
        return None
    if TEACHER_BEST_CHECKPOINT.exists():
        print("Using existing best YOLO26m LoRA teacher:", TEACHER_BEST_CHECKPOINT)
        return TEACHER_BEST_CHECKPOINT
    return None


teacher_checkpoint = select_best_existing_lora_teacher()
if teacher_checkpoint is None:
    teacher_checkpoint = fine_tune_yolo26m_lora_teacher()


Using existing best YOLO26m LoRA teacher: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\checkpoints\teacher_yolo\weights\best.pt


## Distill YOLO26m LoRA into a fresh V3 student

The student starts from random initialization and stays at 199,416 parameters. Dense KD now uses up to three teacher detections that match each ground-truth object, supervising up to nine nearby V3 grid points per detection. Teacher detections are cached at a lower confidence floor; the KD loss ramps in after two GT-only warmup epochs and reaches full weight over six epochs. GT loss remains active in every epoch. EMA weights select and validate the best student. Per-class AP50 and the number of teacher boxes/points used are printed each epoch.


In [7]:
import copy
import gc
import math


def _transform_xywh_rows(rows, scaled_w, scaled_h, left, top, keep_visible=0.4):
    transformed = []
    for row in rows:
        cls, cx, cy, bw, bh = map(float, row[:5])
        extra = list(map(float, row[5:]))
        x1 = cx * scaled_w - left - bw * scaled_w / 2.0
        y1 = cy * scaled_h - top - bh * scaled_h / 2.0
        x2 = cx * scaled_w - left + bw * scaled_w / 2.0
        y2 = cy * scaled_h - top + bh * scaled_h / 2.0
        old_area = max((x2 - x1) * (y2 - y1), 1e-6)
        ix1, iy1 = max(0.0, x1), max(0.0, y1)
        ix2, iy2 = min(float(IMG_W), x2), min(float(IMG_H), y2)
        visible = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1) / old_area
        if visible >= keep_visible and ix2 > ix1 and iy2 > iy1:
            transformed.append([
                cls,
                (ix1 + ix2) / (2.0 * IMG_W),
                (iy1 + iy2) / (2.0 * IMG_H),
                (ix2 - ix1) / IMG_W,
                (iy2 - iy1) / IMG_H,
                *extra,
            ])
    return transformed


class DetDistillDatasetV3(Dataset):
    def __init__(self, paths, teacher_predictions, augment=True):
        self.paths = list(paths)
        self.teacher_predictions = teacher_predictions
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = Image.open(path).convert("RGB").resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)
        label_path = DATA / "train" / "labels" / f"{path.stem}.txt"
        gt_rows = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            values = line.split()
            if len(values) == 5:
                cls, cx, cy, width, height = int(values[0]), *map(float, values[1:])
                if 0 <= cls < NC and width > 0 and height > 0:
                    gt_rows.append([cls, cx, cy, width, height])
        teacher_rows = [row[:] for row in self.teacher_predictions.get(path.stem, [])]

        if self.augment and random.random() < 0.55:
            scale = random.uniform(1.0, 1.65)
            scaled_w, scaled_h = round(IMG_W * scale), round(IMG_H * scale)
            image = image.resize((scaled_w, scaled_h), Image.Resampling.BILINEAR)
            left = random.randint(0, scaled_w - IMG_W)
            top = random.randint(0, scaled_h - IMG_H)
            image = image.crop((left, top, left + IMG_W, top + IMG_H))
            gt_rows = _transform_xywh_rows(gt_rows, scaled_w, scaled_h, left, top)
            teacher_rows = _transform_xywh_rows(teacher_rows, scaled_w, scaled_h, left, top)

        if self.augment and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            gt_rows = [[row[0], 1.0 - row[1], row[2], row[3], row[4]] for row in gt_rows]
            teacher_rows = [[row[0], 1.0 - row[1], row[2], row[3], row[4], row[5]] for row in teacher_rows]

        if self.augment and random.random() < 0.25:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.85, 1.15))
        if self.augment and random.random() < 0.25:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.85, 1.15))

        array = np.asarray(image, dtype=np.uint8).copy()
        tensor = torch.from_numpy(array.transpose(2, 0, 1)).float().div_(255.0)
        gt = torch.tensor(gt_rows, dtype=torch.float32).reshape(-1, 5)
        teacher = torch.tensor(teacher_rows, dtype=torch.float32).reshape(-1, 6)
        return tensor, gt, teacher


def det_distill_collate(batch):
    images, gt_rows, teacher_rows = zip(*batch)
    return torch.stack(images), list(gt_rows), list(teacher_rows)


def _select_student_train_device(min_free_gb=1.25):
    if not torch.cuda.is_available():
        return torch.device("cpu")
    gc.collect()
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    free_gb = free_bytes / (1024 ** 3)
    if free_gb < min_free_gb:
        print(f"Only {free_gb:.2f}/{total_bytes / (1024 ** 3):.2f} GiB GPU memory is free; using CPU for student training to avoid OOM.")
        return torch.device("cpu")
    return torch.device("cuda")


def cache_yolo26m_train_predictions(teacher_checkpoint, train_paths, batch=1, force_cpu=False):
    cache_path = OUTPUT_DIR / "yolo26m_teacher_train_predictions.json"
    source_list_path = OUTPUT_DIR / "teacher_kd_train_images.txt"
    teacher_path = Path(teacher_checkpoint).resolve()
    if not teacher_path.exists():
        raise FileNotFoundError(f"Missing fine-tuned teacher checkpoint: {teacher_path}")
    cache_key = {
        "teacher": str(teacher_path),
        "teacher_size": teacher_path.stat().st_size,
        "teacher_mtime_ns": teacher_path.stat().st_mtime_ns,
        "image_stems": [p.stem for p in train_paths],
        "imgsz": TEACHER_IMGSZ,
        "confidence_floor": 0.01,
        "max_det": 200,
        "nms_iou": 0.70,
    }
    if cache_path.exists():
        cached = json.loads(cache_path.read_text(encoding="utf-8"))
        if cached.get("meta") == cache_key:
            print("Reusing teacher cache:", cache_path)
            return cached["predictions"]

    # A .txt source makes Ultralytics stream the file list in bounded batches.
    # Passing thousands of paths as a Python list can be interpreted as one huge batch.
    source_list_path.write_text("".join(f"{p.resolve()}\n" for p in train_paths), encoding="utf-8")
    teacher = YOLO(str(teacher_path))
    merged_layers = merge_yolo_lora(teacher.model)
    if merged_layers:
        print(f"Merged {merged_layers} LoRA layers for teacher inference.")
    else:
        print("Teacher checkpoint already has plain, merged convolution layers.")

    gc.collect()
    predict_device = "cpu"
    if torch.cuda.is_available() and not force_cpu:
        torch.cuda.empty_cache()
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / (1024 ** 3)
        if free_gb >= 1.5:
            predict_device = 0
        else:
            print(f"Only {free_gb:.2f}/{total_bytes / (1024 ** 3):.2f} GiB GPU memory is free; caching teacher labels on CPU.")

    predictions = {}
    results = teacher.predict(
        source=str(source_list_path),
        imgsz=TEACHER_IMGSZ,
        batch=max(1, int(batch)),
        device=predict_device,
        half=(predict_device != "cpu"),
        conf=0.01,
        iou=0.70,
        max_det=200,
        stream=True,
        verbose=False,
    )
    retry_cpu = False
    try:
        for path, result in zip(train_paths, results):
            rows = []
            if result.boxes is not None and len(result.boxes):
                xyxy = result.boxes.xyxy.detach().cpu().float().numpy()
                cls_ids = result.boxes.cls.detach().cpu().long().numpy()
                scores = result.boxes.conf.detach().cpu().float().numpy()
                for box, cls_id, score in zip(xyxy, cls_ids, scores):
                    x1, y1, x2, y2 = map(float, box)
                    cx, cy = (x1 + x2) / (2.0 * IMG_W), (y1 + y2) / (2.0 * IMG_H)
                    bw, bh = (x2 - x1) / IMG_W, (y2 - y1) / IMG_H
                    rows.append([int(cls_id), cx, cy, bw, bh, float(score)])
            predictions[path.stem] = rows
    except RuntimeError as exc:
        if predict_device != "cpu" and "out of memory" in str(exc).lower():
            print("Teacher-label caching ran out of GPU memory; retrying this cache on CPU.")
            retry_cpu = True
        else:
            raise

    if retry_cpu:
        del results, teacher, predictions
        gc.collect()
        torch.cuda.empty_cache()
        return cache_yolo26m_train_predictions(teacher_checkpoint, train_paths, batch=1, force_cpu=True)
    if len(predictions) != len(train_paths):
        raise RuntimeError(f"Teacher cache is incomplete: got {len(predictions)} predictions for {len(train_paths)} training images.")

    cache_path.write_text(json.dumps({"meta": cache_key, "predictions": predictions}), encoding="utf-8")
    print("Saved teacher predictions for training images:", cache_path)
    del results, teacher
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return predictions

def _xywh_norm_to_xyxy(row):
    cls, cx, cy, bw, bh = map(float, row[:5])
    return cls, (cx - bw / 2.0) * IMG_W, (cy - bh / 2.0) * IMG_H, (cx + bw / 2.0) * IMG_W, (cy + bh / 2.0) * IMG_H


def _iou_xyxy_values(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-7)


def _giou_loss_xyxy(pred, target):
    ix1, iy1 = torch.maximum(pred[0], target[0]), torch.maximum(pred[1], target[1])
    ix2, iy2 = torch.minimum(pred[2], target[2]), torch.minimum(pred[3], target[3])
    inter = (ix2 - ix1).clamp_min(0) * (iy2 - iy1).clamp_min(0)
    area_p = (pred[2] - pred[0]).clamp_min(0) * (pred[3] - pred[1]).clamp_min(0)
    area_t = (target[2] - target[0]).clamp_min(0) * (target[3] - target[1]).clamp_min(0)
    union = (area_p + area_t - inter).clamp_min(1e-6)
    enclosure = (
        (torch.maximum(pred[2], target[2]) - torch.minimum(pred[0], target[0])).clamp_min(0)
        * (torch.maximum(pred[3], target[3]) - torch.minimum(pred[1], target[1])).clamp_min(0)
    ).clamp_min(1e-6)
    iou = inter / union
    giou = iou - (enclosure - union) / enclosure
    return 1.0 - giou


def detector_v3_teacher_kd_loss(
    outputs,
    gt_batch,
    teacher_batch,
    min_teacher_conf=0.10,
    min_gt_iou=0.25,
    max_teacher_per_gt=3,
    max_points_per_box=9,
    return_stats=False,
):
    """Dense, confidence-weighted KD on several student grid points per teacher box."""
    device = outputs[0].device
    kd_cls = torch.zeros((), device=device, dtype=torch.float32)
    kd_ctr = torch.zeros((), device=device, dtype=torch.float32)
    kd_giou = torch.zeros((), device=device, dtype=torch.float32)
    kd_dfl = torch.zeros((), device=device, dtype=torch.float32)
    weight_sum = torch.zeros((), device=device, dtype=torch.float32)
    teacher_boxes_used = 0
    teacher_points_used = 0
    occupied = [set() for _ in teacher_batch]

    for batch_index, (gt_rows, teacher_rows) in enumerate(zip(gt_batch, teacher_batch)):
        if len(gt_rows) == 0 or len(teacher_rows) == 0:
            continue
        gt_xyxy = [_xywh_norm_to_xyxy(row.tolist()) for row in gt_rows]
        gt_teacher_counts = [0] * len(gt_xyxy)
        ordered = sorted(teacher_rows.tolist(), key=lambda row: row[5], reverse=True)

        for row in ordered:
            cls_id, cx, cy, bw, bh, score = map(float, row)
            cls_id = int(cls_id)
            if cls_id < 0 or cls_id >= NC or score < min_teacher_conf:
                continue
            box = (
                (cx - bw / 2.0) * IMG_W,
                (cy - bh / 2.0) * IMG_H,
                (cx + bw / 2.0) * IMG_W,
                (cy + bh / 2.0) * IMG_H,
            )
            if box[2] <= box[0] or box[3] <= box[1]:
                continue

            same_class_matches = [
                (_iou_xyxy_values(box, gt[1:]), gt_index)
                for gt_index, gt in enumerate(gt_xyxy)
                if int(gt[0]) == cls_id
            ]
            if not same_class_matches:
                continue
            match_iou, matched_gt = max(same_class_matches)
            if match_iou < min_gt_iou or gt_teacher_counts[matched_gt] >= max_teacher_per_gt:
                continue

            max_side = max(box[2] - box[0], box[3] - box[1])
            level = 0 if max_side <= LEVEL_MAX_SIZE[0] else 1 if max_side <= LEVEL_MAX_SIZE[1] else 2
            stride = STRIDES[level]
            raw = outputs[level]
            _, _, height, width = raw.shape
            x1, y1, x2, y2 = box
            center_x, center_y = (x1 + x2) / 2.0, (y1 + y2) / 2.0
            radius = 2.0 * stride
            ix0 = max(0, int(math.ceil(x1 / stride - 0.5)))
            ix1 = min(width - 1, int(math.floor(x2 / stride - 0.5)))
            iy0 = max(0, int(math.ceil(y1 / stride - 0.5)))
            iy1 = min(height - 1, int(math.floor(y2 / stride - 0.5)))

            candidates = []
            for iy in range(iy0, iy1 + 1):
                py = (iy + 0.5) * stride
                if abs(py - center_y) > radius:
                    continue
                for ix in range(ix0, ix1 + 1):
                    px = (ix + 0.5) * stride
                    if abs(px - center_x) <= radius:
                        key = (batch_index, level, iy, ix)
                        if key not in occupied[batch_index]:
                            candidates.append(((px - center_x) ** 2 + (py - center_y) ** 2, iy, ix))
            if not candidates:
                ix = max(0, min(width - 1, int(round(center_x / stride - 0.5))))
                iy = max(0, min(height - 1, int(round(center_y / stride - 0.5))))
                key = (batch_index, level, iy, ix)
                if key not in occupied[batch_index]:
                    candidates = [(0.0, iy, ix)]
            if not candidates:
                continue

            candidates = sorted(candidates)[:max_points_per_box]
            point_targets = []
            qualities = []
            for _, iy, ix in candidates:
                anchor_x, anchor_y = (ix + 0.5) * stride, (iy + 0.5) * stride
                ltrb_px = torch.tensor(
                    [anchor_x - x1, anchor_y - y1, x2 - anchor_x, y2 - anchor_y],
                    device=device,
                    dtype=torch.float32,
                ).clamp_min(0.0)
                left, top, right, bottom = ltrb_px.unbind()
                centerness = torch.sqrt(
                    (torch.minimum(left, right) / torch.maximum(left, right).clamp_min(1e-6))
                    * (torch.minimum(top, bottom) / torch.maximum(top, bottom).clamp_min(1e-6))
                ).clamp(0.10, 1.0)
                point_targets.append((iy, ix, anchor_x, anchor_y, ltrb_px / stride))
                qualities.append(centerness)

            quality_sum = torch.stack(qualities).sum().clamp_min(1e-6)
            confidence = torch.tensor(score, device=device, dtype=torch.float32).clamp(0.05, 1.0)
            for (iy, ix, anchor_x, anchor_y, target_ltrb), quality in zip(point_targets, qualities):
                occupied[batch_index].add((batch_index, level, iy, ix))
                point_weight = confidence * quality / quality_sum
                soft_quality = (confidence * quality).clamp(0.0, 1.0)
                logits = raw[batch_index, :, iy, ix].float()
                soft_class = torch.zeros((NC,), device=device, dtype=torch.float32)
                soft_class[cls_id] = soft_quality
                kd_cls = kd_cls + F.binary_cross_entropy_with_logits(
                    logits[:NC], soft_class, reduction="sum"
                ) * point_weight
                kd_ctr = kd_ctr + F.binary_cross_entropy_with_logits(
                    logits[NC + 4 * REG_MAX], soft_quality, reduction="sum"
                ) * point_weight

                target_ltrb = target_ltrb.clamp(0.0, REG_MAX - 1.001)
                dist_logits = logits[NC:NC + 4 * REG_MAX].reshape(4, REG_MAX)
                lo = target_ltrb.floor().long()
                hi = (lo + 1).clamp(max=REG_MAX - 1)
                hi_weight = target_ltrb - lo.float()
                lo_weight = 1.0 - hi_weight
                log_prob = dist_logits.log_softmax(dim=-1)
                dfl_each = -(log_prob.gather(-1, lo[:, None]).squeeze(-1) * lo_weight
                             + log_prob.gather(-1, hi[:, None]).squeeze(-1) * hi_weight).mean()
                kd_dfl = kd_dfl + dfl_each * point_weight

                pred_ltrb = expected_ltrb(raw[batch_index:batch_index + 1], stride)[0, :, iy, ix]
                pred_box = torch.stack((
                    pred_ltrb.new_tensor(anchor_x) - pred_ltrb[0],
                    pred_ltrb.new_tensor(anchor_y) - pred_ltrb[1],
                    pred_ltrb.new_tensor(anchor_x) + pred_ltrb[2],
                    pred_ltrb.new_tensor(anchor_y) + pred_ltrb[3],
                ))
                target_box = torch.tensor(box, device=device, dtype=torch.float32)
                kd_giou = kd_giou + _giou_loss_xyxy(pred_box, target_box) * point_weight

            weight_sum = weight_sum + confidence
            gt_teacher_counts[matched_gt] += 1
            teacher_boxes_used += 1
            teacher_points_used += len(point_targets)

    if weight_sum.item() <= 0:
        loss = outputs[0].new_zeros((), dtype=torch.float32)
    else:
        denom = weight_sum.clamp_min(1e-6)
        loss = kd_cls / denom + 0.5 * kd_ctr / denom + 2.0 * kd_giou / denom + 0.25 * kd_dfl / denom
    stats = {"teacher_boxes": teacher_boxes_used, "teacher_points": teacher_points_used}
    return (loss, stats) if return_stats else loss

class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        self.decay = float(decay)
        self.updates = 0
        for parameter in self.ema.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        decay = self.decay * (1.0 - math.exp(-self.updates / 200.0))
        model_state = model.state_dict()
        for key, ema_value in self.ema.state_dict().items():
            current = model_state[key].detach()
            if ema_value.is_floating_point():
                ema_value.mul_(decay).add_(current, alpha=1.0 - decay)
            else:
                ema_value.copy_(current)


def train_detector_v3_kd(
    teacher_checkpoint=TEACHER_BEST_CHECKPOINT,
    max_epochs=100,
    patience_limit=20,
    batch_size=16,
    seed=42,
    kd_weight=0.55,
    student_lr=1e-3,
    gt_warmup_epochs=2,
    kd_ramp_epochs=6,
    min_teacher_conf=0.10,
    min_gt_iou=0.25,
    max_teacher_per_gt=3,
    max_points_per_box=9,
):
    best_path, last_path = STUDENT_BEST_CHECKPOINT, STUDENT_LAST_CHECKPOINT
    teacher_path_for_resume = Path(teacher_checkpoint).resolve()
    if not teacher_path_for_resume.exists():
        raise FileNotFoundError(f"Missing YOLO26m LoRA teacher checkpoint: {teacher_path_for_resume}")
    teacher_stat = teacher_path_for_resume.stat()
    last_state = _load_checkpoint_file(last_path) if last_path.exists() else None
    last_distillation = last_state.get("distillation", {}) if last_state is not None else {}
    resume_kd = bool(
        last_state is not None
        and last_state.get("training_stage") == "detector_v3_kd"
        and "optimizer" in last_state and "scheduler" in last_state
        and last_distillation.get("teacher") == str(teacher_path_for_resume)
        and last_distillation.get("teacher_size") == teacher_stat.st_size
        and last_distillation.get("teacher_mtime_ns") == teacher_stat.st_mtime_ns
        and last_distillation.get("student_init") == "random_initialization"
        and last_distillation.get("kd_version") == "dense_teacher_v2"
        and float(last_distillation.get("kd_weight", -1.0)) == float(kd_weight)
        and float(last_distillation.get("student_lr", -1.0)) == float(student_lr)
        and int(last_distillation.get("max_epochs", -1)) == int(max_epochs)
        and int(last_distillation.get("gt_warmup_epochs", -1)) == int(gt_warmup_epochs)
        and int(last_distillation.get("kd_ramp_epochs", -1)) == int(kd_ramp_epochs)
        and int(last_distillation.get("seed", -1)) == int(seed)
        and int(last_distillation.get("batch_size", -1)) == int(batch_size)
        and float(last_distillation.get("min_teacher_conf", -1.0)) == float(min_teacher_conf)
        and float(last_distillation.get("min_gt_iou", -1.0)) == float(min_gt_iou)
        and int(last_distillation.get("max_teacher_per_gt", -1)) == int(max_teacher_per_gt)
        and int(last_distillation.get("max_points_per_box", -1)) == int(max_points_per_box)
        and bool(last_distillation.get("student_ema", False))
    )
    if last_state is not None and not resume_kd:
        print("Existing last.pt does not match dense-teacher KD v2; starting a fresh random student.")
        last_state = None

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = _select_student_train_device()
    train_paths, val_paths = detector_train_val_paths(seed)
    teacher_predictions = cache_yolo26m_train_predictions(teacher_checkpoint, train_paths, batch=1)
    train_loader = DataLoader(
        DetDistillDatasetV3(train_paths, teacher_predictions, augment=True),
        batch_size=batch_size, shuffle=True, num_workers=0,
        pin_memory=(device.type == "cuda"), collate_fn=det_distill_collate,
    )
    val_loader = DataLoader(
        DetDatasetV3(val_paths, augment=False), batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=(device.type == "cuda"), collate_fn=det_collate,
    )

    # Stale weights are replaced only after teacher-cache preparation succeeds.
    if not resume_kd:
        for stale_path in (best_path, last_path):
            stale_path.unlink(missing_ok=True)
    model = TinyMultiScaleDetV3().to(device)
    nparams = count_det_parameters(model)
    assert nparams == DET_PARAMS_EXPECTED, (nparams, DET_PARAMS_EXPECTED)
    ema = ModelEMA(model, decay=0.999)
    if resume_kd:
        model.load_state_dict(last_state["model"])
        ema.ema.load_state_dict(last_state.get("ema_model", last_state["model"]))
        ema.updates = int(last_state.get("ema_updates", 0))
        print("Resuming dense-teacher KD V3 from:", last_path)

    optimizer = torch.optim.AdamW(model.parameters(), lr=student_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    if resume_kd:
        optimizer.load_state_dict(last_state["optimizer"])
        scheduler.load_state_dict(last_state["scheduler"])
        if "scaler" in last_state:
            scaler.load_state_dict(last_state["scaler"])
        best_map = float(last_state.get("best_map", -1.0))
        best_epoch = int(last_state.get("best_epoch", 0))
        patience = int(last_state.get("patience", 0))
        history = list(last_state.get("history", []))
        start_epoch = int(last_state["epoch"]) + 1
    else:
        print(
            f"Random V3 student params={nparams:,}; KD weight={kd_weight:g}; "
            f"warmup={gt_warmup_epochs}; ramp={kd_ramp_epochs}; EMA enabled"
        )
        best_map, best_epoch, patience, history, start_epoch = -1.0, 0, 0, [], 1
    if start_epoch > max_epochs or patience >= patience_limit:
        print(f"KD student already reached its stop condition at epoch {start_epoch - 1}.")
        return best_path if best_path.exists() else last_path

    for epoch in range(start_epoch, max_epochs + 1):
        model.train()
        total_loss = total_gt_loss = total_kd_loss = 0.0
        total_teacher_boxes = total_teacher_points = 0
        if epoch <= gt_warmup_epochs:
            kd_ramp = 0.0
        else:
            kd_ramp = min(1.0, (epoch - gt_warmup_epochs) / max(kd_ramp_epochs, 1))

        for images, gt_rows, teacher_rows in train_loader:
            images = images.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                outputs = model(images)
            gt_targets = build_det_targets(gt_rows, outputs, device)
            hard_loss = detector_v3_loss(outputs, gt_targets)
            kd_loss, kd_stats = detector_v3_teacher_kd_loss(
                outputs, gt_rows, teacher_rows,
                min_teacher_conf=min_teacher_conf,
                min_gt_iou=min_gt_iou,
                max_teacher_per_gt=max_teacher_per_gt,
                max_points_per_box=max_points_per_box,
                return_stats=True,
            )
            loss = hard_loss + kd_weight * kd_ramp * kd_loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite KD loss at epoch {epoch}: {loss.item()}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)
            count = len(images)
            total_loss += float(loss.detach()) * count
            total_gt_loss += float(hard_loss.detach()) * count
            total_kd_loss += float(kd_loss.detach()) * count
            total_teacher_boxes += kd_stats["teacher_boxes"]
            total_teacher_points += kd_stats["teacher_points"]

        scheduler.step()
        denom = max(len(train_loader.dataset), 1)
        val_map, per_class_ap = evaluate_det_v3_map50(
            ema.ema, val_loader, device, return_per_class=True
        )
        row = {
            "epoch": epoch,
            "train_loss": total_loss / denom,
            "gt_loss": total_gt_loss / denom,
            "kd_loss": total_kd_loss / denom,
            "kd_ramp": kd_ramp,
            "teacher_boxes_per_image": total_teacher_boxes / denom,
            "teacher_points_per_box": total_teacher_points / max(total_teacher_boxes, 1),
            "val_map50": val_map,
            "lr": scheduler.get_last_lr()[0],
        }
        row.update({f"ap50_{name}": value for name, value in per_class_ap.items()})
        history.append(row)
        improved = val_map > best_map + 1e-5
        if improved:
            best_map, best_epoch, patience = val_map, epoch, 0
        else:
            patience += 1

        distillation = {
            "teacher": str(teacher_path_for_resume),
            "teacher_size": teacher_stat.st_size,
            "teacher_mtime_ns": teacher_stat.st_mtime_ns,
            "student_init": "random_initialization",
            "student_lr": student_lr,
            "max_epochs": max_epochs,
            "gt_warmup_epochs": gt_warmup_epochs,
            "kd_ramp_epochs": kd_ramp_epochs,
            "kd_weight": kd_weight,
            "min_teacher_conf": min_teacher_conf,
            "min_gt_iou": min_gt_iou,
            "max_teacher_per_gt": max_teacher_per_gt,
            "max_points_per_box": max_points_per_box,
            "kd_version": "dense_teacher_v2",
            "student_ema": True,
            "seed": seed,
            "batch_size": batch_size,
        }
        checkpoint = {
            "model": model.state_dict(),
            "ema_model": ema.ema.state_dict(),
            "ema_updates": ema.updates,
            "selected_weights": "ema_student",
            "architecture": "TinyMultiScaleDetV3",
            "training_stage": "detector_v3_kd",
            "epoch": epoch,
            "val_map50": val_map,
            "per_class_ap50": per_class_ap,
            "best_map": best_map,
            "best_epoch": best_epoch,
            "patience": patience,
            "history": history,
            "params": nparams,
            "class_names": CLASS_NAMES,
            "img_size": [IMG_W, IMG_H],
            "strides": list(STRIDES),
            "reg_max": REG_MAX,
            "distillation": distillation,
        }
        if improved or not best_path.exists():
            torch.save({**checkpoint, "model": ema.ema.state_dict()}, best_path)
        torch.save({
            **checkpoint,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
        }, last_path)
        pd.DataFrame(history).to_csv(OUTPUT_DIR / "detector_v3_kd_history.csv", index=False)
        ap_text = " ".join(f"{name}={score:.3f}" for name, score in per_class_ap.items())
        print(
            f"{epoch:03d}: loss={row['train_loss']:.5f} gt={row['gt_loss']:.5f} "
            f"kd={row['kd_loss']:.5f} kd_ramp={kd_ramp:.2f} "
            f"teacher_boxes/img={row['teacher_boxes_per_image']:.2f} "
            f"points/box={row['teacher_points_per_box']:.2f} "
            f"val_mAP50={val_map:.5f} AP50[{ap_text}] "
            f"best={best_map:.5f}@{best_epoch} patience={patience}/{patience_limit}"
        )
        if patience >= patience_limit:
            break

    print(
        f"Best EMA V3 student: {best_path} epoch={best_epoch} "
        f"mAP@0.5={best_map:.5f} params={nparams:,}"
    )
    return best_path


In [ ]:
# Student distillation can use the best completed teacher epoch even if teacher training was interrupted.
# If no best exists yet, use the last completed teacher epoch; the cache helper merges LoRA in memory.
from pathlib import Path
import json

teacher_checkpoint = None
teacher_status = {}
if TEACHER_STATUS_PATH.exists():
    try:
        teacher_status = json.loads(TEACHER_STATUS_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        teacher_status = {}

if TEACHER_BEST_CHECKPOINT.exists():
    teacher_checkpoint = TEACHER_BEST_CHECKPOINT
    if teacher_status.get("completed"):
        print("Using completed best teacher:", teacher_checkpoint)
    else:
        print("Teacher is incomplete/interrupted; using its best saved epoch for KD:", teacher_checkpoint)
elif TEACHER_LAST_CHECKPOINT.exists():
    teacher_checkpoint = TEACHER_LAST_CHECKPOINT
    print("Teacher has no best yet; using last saved epoch for KD:", teacher_checkpoint)
else:
    print("No teacher checkpoint exists; training teacher before starting KD.")
    teacher_checkpoint = fine_tune_yolo26m_lora_teacher()

if not Path(teacher_checkpoint).exists():
    raise FileNotFoundError(f"Missing teacher checkpoint: {teacher_checkpoint}")

detector_v3_kd_checkpoint = train_detector_v3_kd(
    teacher_checkpoint=teacher_checkpoint,
    max_epochs=100,
    kd_weight=0.55,
    gt_warmup_epochs=2,
    kd_ramp_epochs=6,
    min_teacher_conf=0.10,
    min_gt_iou=0.25,
    max_teacher_per_gt=3,
    max_points_per_box=9,
)


Teacher is incomplete/interrupted; using its best saved epoch for KD: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\checkpoints\teacher_yolo\weights\best.pt
Reusing teacher cache: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\output\yolo26m_teacher_train_predictions.json
Resuming dense-teacher KD V3 from: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\checkpoints\student_detector_v3\last.pt
070: loss=2.17397 gt=1.38947 kd=1.42637 kd_ramp=1.00 teacher_boxes/img=6.09 points/box=8.97 val_mAP50=0.95409 AP50[bottle=0.968 cup=0.970 phone=0.939 book=0.949 mouse=0.941 box=0.980 toy_car=0.933] best=0.95437@67 patience=3/20


: 

## Export submission

Load the distilled V3 student and the existing Intent V2 checkpoint, run single-model inference without TTA or ensemble, and save a new submission CSV.

In [ ]:
from PIL import Image
import json
import numpy as np
import pandas as pd
import torch


intent_state = intent_v2_state
detector_ckpt_path = STUDENT_BEST_CHECKPOINT
if not detector_ckpt_path.exists():
    raise FileNotFoundError(f"Run the distillation cell first: {detector_ckpt_path}")
try:
    detector_state = torch.load(detector_ckpt_path, map_location="cpu", weights_only=True)
except TypeError:
    detector_state = torch.load(detector_ckpt_path, map_location="cpu")

infer_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
intent_infer = IntentTextCNNV2().to(infer_device)
intent_infer.load_state_dict(intent_state["model"])
intent_infer.eval()

detector_infer = TinyMultiScaleDetV3().to(infer_device)
detector_infer.load_state_dict(detector_state["model"])
detector_infer.eval()


def predict_intent_v2_submission(text):
    words = torch.tensor(
        [encode_words(text, intent_state["word_vocab"])],
        dtype=torch.long,
        device=infer_device,
    )
    chars = torch.tensor(
        [encode_chars(text)],
        dtype=torch.long,
        device=infer_device,
    )
    with torch.inference_mode():
        scope_logits, action_logits = intent_infer(words, chars)
        if int(scope_logits.argmax(1).item()) == 0:
            return "OUT_OF_SCOPE"
        return INTENT_NAMES[int(action_logits.argmax(1).item())]


def predict_detector_v3_kd(path, conf=0.001, max_det=100):
    image = Image.open(path).convert("RGB").resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.uint8).copy()
    tensor = torch.from_numpy(array.transpose(2, 0, 1)).float().div_(255.0).unsqueeze(0).to(infer_device)
    with torch.inference_mode():
        boxes, scores, classes = decode_det_v3(
            detector_infer(tensor),
            conf=conf,
            max_det=max_det,
        )[0]
    records = []
    for box, score, cls in zip(boxes, scores, classes):
        records.append({
            "class_id": int(cls),
            "score": round(float(score), 6),
            "bbox": [round(float(value), 2) for value in box.tolist()],
        })
    return json.dumps(records, separators=(",", ":"))


intent_test = pd.read_csv(DATA / "intent_test.csv")
intent_predictions = [predict_intent_v2_submission(question) for question in intent_test.question]
detection_predictions = {
    path.stem: predict_detector_v3_kd(path)
    for path in sorted((DATA / "test" / "images").glob("*.png"))
}

submission = pd.read_csv(DATA / "sample_submission.csv")
intent_by_id = dict(zip(intent_test.question_id, intent_predictions))
submission["prediction"] = submission.apply(
    lambda row: (
        detection_predictions[row.sample_id]
        if row.task_type == "DETECTION"
        else intent_by_id[row.sample_id]
    ),
    axis=1,
)
submission_path = OUTPUT_DIR / "submission_v3_kd_lora.csv"
submission.to_csv(submission_path, index=False)
print("Saved submission:", submission_path, submission.shape)